## Load NAB time series 

## Step 1 - Load CPU dataset and attach anomaly labels

In this step, the raw `cpu_utilization_asg_misconfiguration.csv` file from the NAB `realKnownCause` folder is loaded into a dataframe. The corresponding anomaly timestamps are retrieved from `combined_labels.json` (using the key `realKnownCause/cpu_utilization_asg_misconfiguration.csv`) and converted into a binary `is_anomaly` column. The `timestamp` column is parsed into a proper datetime type and the rows are sorted chronologically so that each record reflects the correct order in time. Finally, a class-balance table is produced to confirm the number and proportion of normal vs anomalous points before moving into schema standardisation.


### A) Load raw cpu_utilization_asg_misconfiguration.csv dataset

In [10]:
# Imports
import pandas as pd
import numpy as np


In [11]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load file
file_path = os.path.join(base_path, "cpu_utilization_asg_misconfiguration.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2014-05-14 01:14:00,85.835
1,2014-05-14 01:19:00,88.167
2,2014-05-14 01:24:00,44.595
3,2014-05-14 01:29:00,56.282
4,2014-05-14 01:34:00,36.534


### B) Load Anomaly Labels (JSON)

In [12]:
import json  

# Where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# Show the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]

['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

### C) Extract Labels for Selected Dataset

In [17]:
# Select the correct dataset key from the labels dictionary
dataset_key = "realKnownCause/cpu_utilization_asg_misconfiguration.csv"

# Extract the list of anomaly timestamps for this dataset
cpu_label_times = labels[dataset_key]

# Turn the list of timestamp strings into a DataFrame
cpu_label_times_df = pd.DataFrame(
    {"timestamp": pd.to_datetime(cpu_label_times)}
)

# Display the timestamps
cpu_label_times_df.head()

,timestamp
0,2014-07-12 02:04:00
1,2014-07-14 21:44:00


### D) Add Labels to DataFrame

In [15]:
# Converting the label timestamps into datetime objects 
cpu_label_times = pd.to_datetime(labels[dataset_key])

# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Create a new column: 1 if the timestamp is an anomaly, 0 otherwise
df["is_anomaly"] = df["timestamp"].isin(cpu_label_times).astype(int)

# See the updated dataframe
df.head()


,timestamp,value,is_anomaly
0,2014-05-14 01:14:00,85.835,0
1,2014-05-14 01:19:00,88.167,0
2,2014-05-14 01:24:00,44.595,0
3,2014-05-14 01:29:00,56.282,0
4,2014-05-14 01:34:00,36.534,0


### E) Class Balance

In [16]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,18048,99.989
1,Anomaly,2,0.011


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [18]:
# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Rename to common research project schema
df = df.rename(columns={"timestamp": "time"})

# Add a case_study identifier
df["case_study"] = "cpu"

# Quick check
df[["time", "value", "is_anomaly", "case_study"]].head()


,time,value,is_anomaly,case_study
0,2014-05-14 01:14:00,85.835,0,cpu
1,2014-05-14 01:19:00,88.167,0,cpu
2,2014-05-14 01:24:00,44.595,0,cpu
3,2014-05-14 01:29:00,56.282,0,cpu
4,2014-05-14 01:34:00,36.534,0,cpu


###  Quick verification

In [19]:
df[["time","value","is_anomaly","case_study"]].dtypes

time          datetime64[ns]
value                float64
is_anomaly             int64
case_study            object
dtype: object


In this step, the CPU utilisation dataframe is aligned with the common schema used across all case studies. The `timestamp` column is converted to a proper datetime type, the rows are sorted chronologically, and the column is renamed to `time`. A `case_study` column with the value `"cpu"` is added so that this dataset can later be stacked or filtered alongside the other case studies using the same core columns (`time`, `value`, `is_anomaly`, `case_study`).


In [22]:
# Step 3A – Missing values sanity check

missing_values_df = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

missing_values_df.columns = ["column_name", "missing_count"]
missing_values_df


,column_name,missing_count
0,time,0
1,value,0
2,is_anomaly,0
3,case_study,0
